In [1]:
import pyrosetta
from pyrosetta.rosetta.core.select.residue_selector import ChainSelector, NeighborhoodResidueSelector, OrResidueSelector
from pyrosetta.rosetta.core.select.movemap import MoveMapFactory, move_map_action
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

# 1. Initialize PyRosetta (Forced 1 repeat at the global level for speed)
pyrosetta.init("-relax:default_repeats 1 -ex1 -ex2aro -ignore_unrecognized_res 1")

# 2. Load your AlphaFold .cif model
cif_path = "/home/tonypeonio/ProteinDesignChallenge/pyrosetta_outputs/fold_38_30_72_decent/fold_38_30_72_decent_model_0.cif"
print(f"Loading pose from: {cif_path}")
pose = pyrosetta.pose_from_file(cif_path)
scorefxn = pyrosetta.create_score_function("ref2015")

# 3. Create the Bubble Selector for Chains E, F, and G
print("Setting up localized MoveMap (Bubble relaxation around E, F, G)...")
sel_E = ChainSelector("E")
sel_F = ChainSelector("F")
sel_G = ChainSelector("G")

# OrResidueSelector only takes two arguments at a time, so we chain them together:
sel_EF = OrResidueSelector(sel_E, sel_F)
sel_EFG = OrResidueSelector(sel_EF, sel_G)

# Create a 10.0 Angstrom bubble around E, F, and G (True means it includes E, F, G themselves)
move_zone = NeighborhoodResidueSelector(sel_EFG, 10.0, True)

# 4. Apply the MoveMap
mm_factory = MoveMapFactory()
# Freeze everything by default
mm_factory.all_bb(False)
mm_factory.all_chi(False)
mm_factory.all_jumps(False)
# Unfreeze ONLY our 10.0 Angstrom bubble
mm_factory.add_chi_action(move_map_action.mm_enable, move_zone)
mm_factory.add_bb_action(move_map_action.mm_enable, move_zone)
movemap = mm_factory.create_movemap_from_pose(pose)

# 5. Run the targeted FastRelax
print("Running targeted 1-repeat FastRelax... (Should be much faster!)")
relax = FastRelax(scorefxn, 1) # Explicitly 1 repeat
relax.set_movemap(movemap)
relax.apply(pose)

# 6. Measure binding energies (ddG) for your analysis
print("\nRelaxation complete! Calculating thermodynamic energies (ddG)...")

iam_E = InterfaceAnalyzerMover("E_ABCDFG")
iam_E.apply(pose)
ddg_E = iam_E.get_interface_dG()

iam_F = InterfaceAnalyzerMover("F_ABCDEG")
iam_F.apply(pose)
ddg_F = iam_F.get_interface_dG()

iam_G = InterfaceAnalyzerMover("G_ABCDEF")
iam_G.apply(pose)
ddg_G = iam_G.get_interface_dG()

# 7. Print the final results
print("\n" + "="*50)
print("LOCALIZED BINDING ENERGY (ddG) RESULTS")
print("="*50)
print(f"Chain E (Touching Binder):  {ddg_E:>8.2f} REU")
print(f"Chain F (Isolated Native):  {ddg_F:>8.2f} REU")
print(f"Chain G (Your Binder):      {ddg_G:>8.2f} REU")
print("-" * 50)

# Quick mathematical breakdown
ddg_diff = abs(ddg_E - ddg_F)
print(f"Difference between native chains E and F: {ddg_diff:.2f} REU\n")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2026.06+release.1a56185c2592611dec4c9c75ddc9468cd2227c1f 2026-01-30T13:14:27] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.conda.ubuntu.cxx11th

In [2]:
pose.dump_pdb("../pyrosetta_outputs/relaxed_decent.pdb")

True